# 04 — Chạy CCSD(T) để lấy nhãn năng lượng, rồi dựng dataset
CCSD(T)/cc-pVDZ toàn electron trên cùng hình học (RHF→CCSD(T) cho phân tử, UHF→UCCSD(T) cho nguyên tử). Nhãn học là **Δ = E_CCSD(T) − E_PBE[n_PBE]** (cùng hình học, cùng basis); đây là hiệu chỉnh năng lượng tổng chứ không phải E_xc trực tiếp.

- Có **resume** (bỏ qua hệ đã có file nhãn) và dừng nhận việc mới sau `MAX_WALL_H` giờ.
- Cuối notebook: ghép mật độ + nhãn thành `data/processed/qm9_xc_dataset.h5` kèm manifest/metadata và in kiểm tra nhanh.

**Đầu ra:** `data/processed/labels/<name>.json`, `qm9_xc_dataset.h5`, `dataset_manifest.csv`.

In [ ]:
# Cài thư viện (chỉ trên Colab; runtime Colab bị reset sau mỗi phiên). Chạy cục bộ: pip install -r requirements.txt
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf h5py

In [ ]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ml_xc_colab")
else:                                    # chạy cục bộ (VS Code/Jupyter): notebook nằm trong notebooks/
    here = Path(os.environ.get("ML_XC_ROOT", Path.cwd())).resolve()
    ROOT = next((p for p in (here, here.parent) if (p / "utils").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Không thấy thư mục utils/. Mở notebook trong ml_xc_colab/notebooks hoặc đặt ML_XC_ROOT.")
sys.path.insert(0, str(ROOT))
ROOT = str(ROOT)
DATA, RAW, PROC, CKPT = (f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed", f"{ROOT}/data/checkpoints")
for d in (RAW, PROC, CKPT):
    os.makedirs(d, exist_ok=True)
try:                                     # giới hạn bộ nhớ PySCF = 70% RAM (MB); đặt trước khi import pyscf
    import psutil
    os.environ.setdefault("PYSCF_MAX_MEMORY", str(int(psutil.virtual_memory().total / 1e6 * 0.7)))
except ImportError:
    pass
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
from utils import dft_utils as dft
from utils import data_utils as du

BASIS = "cc-pvdz"
LAB_DIR, PBE_DIR = f"{PROC}/labels", f"{PROC}/pbe"
os.makedirs(LAB_DIR, exist_ok=True)
FORCE_RERUN = False
MAX_WALL_H = 10.5

systems = pd.read_csv(f"{PROC}/systems.csv")
systems = systems.sort_values(["is_atom", "n_atoms"], ascending=[False, True])
todo = [r for r in systems.itertuples() if FORCE_RERUN or not os.path.exists(f"{LAB_DIR}/{r.name}.json")]
print(f"{len(systems)} hệ | đã có nhãn {len(systems) - len(todo)} | sẽ chạy {len(todo)}")

In [ ]:
t_start = time.time()
n_new = n_fail = 0
for r in todo:
    if (time.time() - t_start) / 3600 > MAX_WALL_H:
        print("Dừng theo giới hạn thời gian; chạy lại notebook để tiếp tục.")
        break
    try:
        mol = dft.build_mol(json.loads(r.symbols_json), json.loads(r.coords_json), basis=BASIS, spin=int(r.spin))
        lab = dft.run_ccsdt(mol)
        lab.update(name=r.name, basis=BASIS, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
        json.dump(lab, open(f"{LAB_DIR}/{r.name}.json", "w"), indent=2)
        n_new += 1
        ok = lab["hf_converged"] and lab["ccsd_converged"]
        print(f"{r.name:14s} {r.formula:10s} nbf={lab['nbf']:3d}  E_CCSD(T)={lab['e_ccsdt']:.6f}  (T)={lab['e_t']*1e3:.2f} mHa  "
              f"conv={ok}  {lab['wall_s']}s{'' if ok else '  <-- KIỂM TRA'}")
    except Exception as e:
        n_fail += 1
        print(f"{r.name:14s} {r.formula:10s} LỖI: {type(e).__name__}: {str(e)[:120]}")
print(f"\nXong lượt này: mới {n_new}, lỗi {n_fail} | {(time.time()-t_start)/60:.1f} phút")

## Dựng dataset HDF5
Hệ không hội tụ hoặc thiếu file bị loại, lý do ghi vào manifest.

In [ ]:
H5 = f"{PROC}/qm9_xc_dataset.h5"
meta = {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "basis": BASIS, "seed": json.load(open(f"{PROC}/selection_report.json")).get("seed"),
        "label": "delta = E_CCSD(T) - E_PBE[n_PBE]; PBE tự hợp, cùng hình học QM9 và basis",
        "grid_level": json.load(open(f"{PBE_DIR}/{systems.name.iloc[0]}.json")).get("grid_level")}
import pyscf; meta["pyscf"] = pyscf.__version__
manifest = du.build_h5(H5, systems, PBE_DIR, LAB_DIR, meta)
manifest.merge(systems[["name", "formula", "smiles", "gdb_id", "n_atoms", "n_heavy", "is_atom"]], on="name").to_csv(
    f"{PROC}/dataset_manifest.csv", index=False)
print(f"Đã lưu {H5} ({os.path.getsize(H5)/1e6:.1f} MB) | đưa vào: {int(manifest.included.sum())}/{len(manifest)}")
bad = manifest[~manifest.included]
print("Loại:", bad[["name", "reason"]].to_dict("records") if len(bad) else "không có")

## Kiểm tra nhanh nhãn
Δ tăng theo kích thước phân tử (mức vài trăm mHa), nên tổng năng lượng DFT và CCSD(T) không so sánh trực tiếp được; sai số của PBE trên năng lượng nguyên tử hóa mới là thước đo có ý nghĩa.

In [ ]:
tab = du.energy_table(H5)
mols = tab[~tab.is_atom]
print("Δ = E_CCSD(T) − E_PBE (Ha):  trung bình", round(mols.delta.mean(), 4), "| min", round(mols.delta.min(), 4), "| max", round(mols.delta.max(), 4))
print("Δ theo nguyên tử (mHa/nguyên tử):", round((mols.delta / mols.n_atoms).mean() * 1e3, 2), "±", round((mols.delta / mols.n_atoms).std() * 1e3, 2))
err = du.ae_error_kcal(tab, list(mols.index), "e_pbe")
print("PBE − CCSD(T) trên năng lượng nguyên tử hóa (kcal/mol):", {k: round(v, 2) for k, v in du.error_stats(err).items()})